# Part 3: Damned if you do, damned if you don't (Julia Implementation)

This notebook implements the "damned if you do, damned if you don't" example from Lab7 and extends it with additional analysis using Julia.

In [ ]:
# Import required packages
using Random, Distributions
using DataFrames, CSV
using Plots, StatsPlots
using GLM, StatsBase
using CausalInference
using LinearAlgebra
using Combinatorics

# Set random seed for reproducibility
Random.seed!(42)

# Create output directory if it doesn't exist
output_dir = "../output"
if !isdir(output_dir)
    mkpath(output_dir)
end

# Set plotting backend
gr()

## Original "Damned if you do, damned if you don't" Example

From Lab7 Example 4: This represents a situation where it's unclear whether Z should be used as a control.

In [ ]:
# Create the original DAG from Lab7 Example 4
# Variables: [X, Z, Y]
original_dag_adj = [0 0 1;  # X -> Y
                    0 0 1;  # Z -> Y
                    0 0 0]  # Y -> none

println("Original DAG: Damned if you do, damned if you don't")
println("Structure:")
println("X -> Y")
println("Z -> Y")
println("X and Z are independent (no direct connection)")

In [ ]:
# Simulate data for original example (following Lab7 structure)
n = 10000

# Generate data as in Lab7
Z_orig = rand(Normal(0, 1), n)
X_orig = rand(Normal(0, 1), n)  # X independent of Z
Y_orig = X_orig .+ Z_orig .+ rand(Normal(0, 1), n)  # Both X and Z affect Y

# True causal effect of X on Y is 1.0
true_effect = 1.0

println("Original example data generated. True causal effect of X on Y: $true_effect")
println("Sample size: $n")

In [ ]:
# Run regressions with and without Z control
df_orig = DataFrame(X = X_orig, Z = Z_orig, Y = Y_orig)

# Regression 1: Y vs X (without Z)
model1_orig = lm(@formula(Y ~ X), df_orig)
coef1_orig = coef(model1_orig)[2]
ci1_orig = confint(model1_orig, level=0.99)[2, :]

# Regression 2: Y vs X, Z (with Z)
model2_orig = lm(@formula(Y ~ X + Z), df_orig)
coef2_orig = coef(model2_orig)[2]  # Coefficient for X
ci2_orig = confint(model2_orig, level=0.99)[2, :]  # 99% CI for X

println("Original Example Results:")
println("Without Z control: $(round(coef1_orig, digits=4)) [99% CI: $(round(ci1_orig[1], digits=4)), $(round(ci1_orig[2], digits=4))]")
println("With Z control:    $(round(coef2_orig, digits=4)) [99% CI: $(round(ci2_orig[1], digits=4)), $(round(ci2_orig[2], digits=4))]")
println("True effect:       $(round(true_effect, digits=4))")

In [ ]:
# Plot coefficients for original example
orig_results = DataFrame(
    Model = ["Without Z", "With Z"],
    Coefficient = [coef1_orig, coef2_orig],
    CI_Lower = [ci1_orig[1], ci2_orig[1]],
    CI_Upper = [ci1_orig[2], ci2_orig[2]]
)

error_lower = orig_results.Coefficient .- orig_results.CI_Lower 
error_upper = orig_results.CI_Upper .- orig_results.Coefficient

p_orig = scatter([1, 2], orig_results.Coefficient,
                yerror=(error_lower, error_upper),
                xlabel="Model Specification",
                ylabel="Estimated Effect of X on Y",
                title="Original Example: Effect Estimates with 99% Confidence Intervals",
                xticks=([1, 2], orig_results.Model),
                markersize=8,
                size=(600, 400),
                legend=false)

# Add true effect line
hline!(p_orig, [true_effect], color=:red, linestyle=:dash, linewidth=2)
annotate!(p_orig, 1, true_effect + 0.05, text("True Effect = $true_effect", :red, 10))

display(p_orig)
savefig(p_orig, joinpath(output_dir, "part3_original_coefficients_Julia.png"))

## Modified Example: Z also affects X

Now we modify the DAG so that Z also has an effect on X, and we can observe additional variables U1 and U2.

In [ ]:
# Create modified DAG where Z affects X, and add U1 and U2
# Variables: [U1, U2, Z, X, Y] (indexed 1-5) 
modified_dag_adj = [0 0 0 1 1;  # U1 -> X, Y
                    0 0 1 0 1;  # U2 -> Z, Y
                    0 0 0 1 1;  # Z -> X, Y
                    0 0 0 0 1;  # X -> Y
                    0 0 0 0 0]  # Y -> none

modified_variables = ["U1", "U2", "Z", "X", "Y"]

println("Modified DAG: Z affects X, with observable U1 and U2")
println("Structure:")
for i in 1:5, j in 1:5
    if modified_dag_adj[i,j] == 1
        println("$(modified_variables[i]) -> $(modified_variables[j])")
    end
end

In [ ]:
# Simulate data for modified example
n = 10000

# Generate exogenous variables
U1 = rand(Normal(0, 1), n)
U2 = rand(Normal(0, 1), n)
eps_Z = rand(Normal(0, 1), n)
eps_X = rand(Normal(0, 1), n)
eps_Y = rand(Normal(0, 1), n)

# Generate endogenous variables following DAG structure
Z = U2 .+ eps_Z          # U2 -> Z
X = U1 .+ Z .+ eps_X      # U1 -> X, Z -> X  
Y = X .+ U1 .+ U2 .+ Z .+ eps_Y  # X -> Y, U1 -> Y, U2 -> Y, Z -> Y

# Create DataFrame
df_modified = DataFrame(
    U1 = U1,
    U2 = U2,
    Z = Z,
    X = X,
    Y = Y
)

println("Modified example data generated.")
println("True causal effect of X on Y: $true_effect")
println("Sample size: $n")
println("\nData summary:")
println(describe(df_modified))

## Comprehensive Regression Analysis

We'll run all possible combinations of controls from {Z, U1, U2}, resulting in 2³ = 8 regressions.

In [ ]:
# Generate all possible combinations of controls
controls = ["Z", "U1", "U2"]

# Generate all subsets (power set)
function powerset(s)
    result = [[]]
    for elem in s
        result = vcat(result, [vcat(subset, [elem]) for subset in result])
    end
    return result
end

all_combinations = powerset(controls)

println("All combinations of controls:")
for (i, combo) in enumerate(all_combinations)
    if isempty(combo)
        println("$i: None")
    else
        println("$i: $(join(combo, ", "))")
    end
end
println("Total number of regressions: $(length(all_combinations))")

In [ ]:
# Run all regressions
results_modified = DataFrame(
    Controls = String[],
    Beta = Float64[],
    SE = Float64[],
    Bias = Float64[]
)

for controls_subset in all_combinations
    # Prepare regression formula
    if isempty(controls_subset)
        formula_str = @formula(Y ~ X)
        control_names = "None"
    else
        # Build formula dynamically
        controls_str = join(controls_subset, " + ")
        formula_expr = Meta.parse("@formula(Y ~ X + $controls_str)")
        formula_str = eval(formula_expr)
        control_names = join(controls_subset, ", ")
    end
    
    # Run regression
    model = lm(formula_str, df_modified)
    
    # Extract results for X (coefficient index 2, after intercept)
    coef_X = coef(model)[2]
    se_X = stderror(model)[2]
    
    push!(results_modified, (control_names, coef_X, se_X, coef_X - true_effect))
end

println("All regressions completed.")

In [ ]:
# Display and save results table
results_display = copy(results_modified)
results_display.Beta = round.(results_display.Beta, digits=4)
results_display.SE = round.(results_display.SE, digits=4)
results_display.Bias = round.(results_display.Bias, digits=4)

println("\nResults Table (Effect of X on Y):")
println("="^50)
println(results_display)
println("="^50)
println("True effect: $(round(true_effect, digits=3))")

In [ ]:
# Save results table in different formats
# Save as CSV
CSV.write(joinpath(output_dir, "part3_results_table_Julia.csv"), results_modified)

# Save as formatted text
open(joinpath(output_dir, "part3_results_table_Julia.txt"), "w") do f
    println(f, "Results Table: Effect of X on Y")
    println(f, "="^40)
    println(f, results_display)
    println(f, "\nTrue causal effect: $(round(true_effect, digits=3))")
end

println("\nResults table saved in multiple formats:")
println("- part3_results_table_Julia.csv")
println("- part3_results_table_Julia.txt")

## Analysis and Conclusions

In [ ]:
# Identify which estimates are closest to the true effect
results_modified.abs_bias = abs.(results_modified.Bias)
sorted_results = sort(results_modified, :abs_bias)
best_estimates = sorted_results[1:3, :]

println("\nBest Estimates (lowest absolute bias):")
for col in ["Beta", "SE", "Bias"]
    best_estimates[!, col] = round.(best_estimates[!, col], digits=4)
end
println(best_estimates[:, ["Controls", "Beta", "SE", "Bias"]])

# Find minimal sufficient set
tolerance = 0.05  # Consider estimates within 0.05 of true effect as "good"
good_estimates = filter(row -> row.abs_bias < tolerance, results_modified)

println("\nEstimates within $tolerance of true effect:")
if nrow(good_estimates) > 0
    good_display = copy(good_estimates)
    for col in ["Beta", "SE", "Bias"]
        good_display[!, col] = round.(good_display[!, col], digits=4)
    end
    println(good_display[:, ["Controls", "Beta", "SE", "Bias"]])
    
    # Find minimal set (fewest controls)
    good_estimates.num_controls = [r == "None" ? 0 : length(split(r, ", ")) for r in good_estimates.Controls]
    minimal_idx = argmin(good_estimates.num_controls)
    minimal_set = good_estimates[minimal_idx, :]
    
    println("\nMinimal sufficient set: $(minimal_set.Controls)")
    println("Estimate: $(round(minimal_set.Beta, digits=4)) (SE: $(round(minimal_set.SE, digits=4)))")
else
    println("No estimates within tolerance found.")
end

## Answers to Questions

### In what way(s) can you get a good estimate of the causal effect?

Based on the results above, good estimates can be obtained by:

1. **Controlling for U1 and U2**: These block the backdoor paths through the confounders
2. **The combination that includes both U1 and U2** provides the most unbiased estimate
3. **Controlling for Z alone may not be sufficient** because Z is now on the causal path (mediator) and also a confounder

### What is the minimal sufficient set of controls?

The minimal sufficient set appears to be **{U1, U2}** because:
- U1 is a confounder (affects both X and Y)
- U2 is a confounder (affects Z which affects both X and Y)
- Controlling for both blocks all backdoor paths from X to Y

### Intuition for why these controls provide good estimates

The intuition is based on blocking backdoor paths:

1. **U1 → X ← Z → Y**: U1 confounds X and Y directly
2. **U2 → Z → X** and **U2 → Z → Y**: U2 confounds X and Y through Z
3. **By controlling for U1 and U2**, we block these confounding paths
4. **Z is problematic** because it's both a confounder and mediator - controlling for it blocks some confounding but also blocks part of the causal effect
5. **The backdoor criterion** suggests controlling for {U1, U2} satisfies the conditions for causal identification

In [ ]:
# Save all data
CSV.write(joinpath(output_dir, "part3_modified_data_Julia.csv"), df_modified)

println("\nAll files saved to output directory:")
println("- part3_original_coefficients_Julia.png")
println("- part3_results_table_Julia.csv")
println("- part3_results_table_Julia.txt")
println("- part3_modified_data_Julia.csv")
println("\nPart 3 analysis complete (Julia implementation)!")